In [4]:
from langchain_groq import ChatGroq
from langchain.prompts import PromptTemplate

# Use a supported model instead of the deprecated one
llm = ChatGroq(
    api_key="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",
    model="llama-3.1-8b-instant"
)

prompt = PromptTemplate(
    input_variables=["question"],
    template="Answer clearly: {question}"
)
chain = prompt | llm

response = chain.invoke({"question": "Who was the first person to walk on the moon?"})
print(response.content)


The first person to walk on the moon was Neil Armstrong. He stepped onto the lunar surface on July 20, 1969, as part of the Apollo 11 mission.


In [5]:
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader("https://careers.nike.com/data-engineer-itc/job/R-70937")
page_data=loader.load().pop().page_content
print(page_data)

USER_AGENT environment variable not set, consider setting it to identify your requests.






















Data Engineer, ITC










































Skip to main content
Open Virtual Assistant










Home


Career Areas


Total Rewards


Life@Nike


Purpose










Language





Select a Language

  Deutsch  
  English  
  Español (España)  
  Español (América Latina)  
  Français  
  Italiano  
  Nederlands  
  Polski  
  Tiếng Việt  
  Türkçe  
  简体中文  
  繁體中文  
  עִברִית  
  한국어  
  日本語  








Careers


















Close Menu







Careers






Chat






                                Home
                            



                                Career Areas
                            



                                Total Rewards
                            



                                Life@Nike
                            



                                Purpose
                            










Jordan Careers







Converse Careers










Language











Menu



Return to Previous Menu



Select a

In [6]:
from langchain_core.prompts import PromptTemplate

promt_extract=PromptTemplate.from_template(
    """
        ### SCRAPED TEXT FROM WEBSITE:
        {page_data}
        ### INSTRUCTION:
        The scraped text is from the career's page of a website.
        Your job is to extract the job postings and return them in JSON format containing the 
        following keys: `role`, `experience`, `skills` and `description`.
        Only return the valid JSON.
        ### VALID JSON (NO PREAMBLE):    
        """
)
chain_extract=promt_extract|llm
res=chain_extract.invoke(input={"page_data":page_data})
print(res.content)

{
  "Data Engineer, ITC": {
    "role": "Data Engineer, ITC",
    "experience": "Over 3 years of industry experience in data engineering",
    "skills": [
      "Proficient in SQL, Spark, and Python",
      "Practical experience with Databricks, AWS, and Snowflake platforms",
      "Solid understanding of data modeling, ETL processes, and data streaming technologies",
      "Familiarity with AWS services including Lambda, Step Functions, DynamoDB, Elasticsearch, and S3",
      "Experience programming in Java, with knowledge of JavaScript/TypeScript and Node.js",
      "Skilled in developing data visualization dashboards using frameworks such as React, Vue.js, or Angular",
      "Experienced in API development leveraging REST and GraphQL APIs for data services",
      "Strong grasp of microservices architecture and architectural design patterns",
      "Comprehensive understanding of CI/CD practices and DevOps responsibilities related to deploying and maintaining production software",
 

In [7]:
type(res.content)

str

In [9]:
from langchain_core.output_parsers import JsonOutputParser

Json_parser=JsonOutputParser()
json_res=Json_parser.parse(res.content)
json_res

{'Data Engineer, ITC': {'role': 'Data Engineer, ITC',
  'experience': 'Over 3 years of industry experience in data engineering',
  'skills': ['Proficient in SQL, Spark, and Python',
   'Practical experience with Databricks, AWS, and Snowflake platforms',
   'Solid understanding of data modeling, ETL processes, and data streaming technologies',
   'Familiarity with AWS services including Lambda, Step Functions, DynamoDB, Elasticsearch, and S3',
   'Experience programming in Java, with knowledge of JavaScript/TypeScript and Node.js',
   'Skilled in developing data visualization dashboards using frameworks such as React, Vue.js, or Angular',
   'Experienced in API development leveraging REST and GraphQL APIs for data services',
   'Strong grasp of microservices architecture and architectural design patterns',
   'Comprehensive understanding of CI/CD practices and DevOps responsibilities related to deploying and maintaining production software',
   'Knowledgeable in implementing and integr

In [10]:
type(json_res)

dict

In [11]:
import pandas as pd
df=pd.read_csv('my_portfolio.csv')
df

,Techstack,Links
0,"React, Node.js, MongoDB",https://example.com/react-portfolio
1,"Angular,.NET, SQL Server",https://example.com/angular-portfolio
2,"Vue.js, Ruby on Rails, PostgreSQL",https://example.com/vue-portfolio
3,"Python, Django, MySQL",https://example.com/python-portfolio
4,"Java, Spring Boot, Oracle",https://example.com/java-portfolio
5,"Flutter, Firebase, GraphQL",https://example.com/flutter-portfolio
6,"WordPress, PHP, MySQL",https://example.com/wordpress-portfolio
7,"Magento, PHP, MySQL",https://example.com/magento-portfolio
8,"React Native, Node.js, MongoDB",https://example.com/react-native-portfolio
9,"iOS, Swift, Core Data",https://example.com/ios-portfolio


In [12]:
import uuid
import chromadb

client = chromadb.PersistentClient('vectorstore')
collection = client.get_or_create_collection(name="portfolio")

if not collection.count():
    for _, row in df.iterrows():
        collection.add(documents=row["Techstack"],
                       metadatas={"links": row["Links"]},
                       ids=[str(uuid.uuid4())])

In [14]:
links = collection.query(query_texts=['Experience in python','Experience in react'], n_results=2)
links

{'ids': [['cad1bc87-5830-4230-a046-c320ac7d3050',
   '82b8aa0a-4fca-4a4f-a15d-4bb80047df1c'],
  ['f3a744e2-9c8e-41a8-a357-899662094963',
   '7c16f0b4-9815-4d23-953b-f2421aac628d']],
 'embeddings': None,
 'documents': [['Machine Learning, Python, TensorFlow',
   'Python, Django, MySQL'],
  ['React, Node.js, MongoDB', 'React Native, Node.js, MongoDB']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[{'links': 'https://example.com/ml-python-portfolio'},
   {'links': 'https://example.com/python-portfolio'}],
  [{'links': 'https://example.com/react-portfolio'},
   {'links': 'https://example.com/react-native-portfolio'}]],
 'distances': [[0.996884822845459, 1.0578701496124268],
  [0.9630732536315918, 1.0685749053955078]]}

In [15]:
links = collection.query(query_texts=['Experience in python','Experience in react'], n_results=2).get('metadatas', [])
links

[[{'links': 'https://example.com/ml-python-portfolio'},
  {'links': 'https://example.com/python-portfolio'}],
 [{'links': 'https://example.com/react-portfolio'},
  {'links': 'https://example.com/react-native-portfolio'}]]

In [20]:
job=json_res
job

{'Data Engineer, ITC': {'role': 'Data Engineer, ITC',
  'experience': 'Over 3 years of industry experience in data engineering',
  'skills': ['Proficient in SQL, Spark, and Python',
   'Practical experience with Databricks, AWS, and Snowflake platforms',
   'Solid understanding of data modeling, ETL processes, and data streaming technologies',
   'Familiarity with AWS services including Lambda, Step Functions, DynamoDB, Elasticsearch, and S3',
   'Experience programming in Java, with knowledge of JavaScript/TypeScript and Node.js',
   'Skilled in developing data visualization dashboards using frameworks such as React, Vue.js, or Angular',
   'Experienced in API development leveraging REST and GraphQL APIs for data services',
   'Strong grasp of microservices architecture and architectural design patterns',
   'Comprehensive understanding of CI/CD practices and DevOps responsibilities related to deploying and maintaining production software',
   'Knowledgeable in implementing and integr

In [22]:
skills=job['Data Engineer, ITC']['skills']
skills

['Proficient in SQL, Spark, and Python',
 'Practical experience with Databricks, AWS, and Snowflake platforms',
 'Solid understanding of data modeling, ETL processes, and data streaming technologies',
 'Familiarity with AWS services including Lambda, Step Functions, DynamoDB, Elasticsearch, and S3',
 'Experience programming in Java, with knowledge of JavaScript/TypeScript and Node.js',
 'Skilled in developing data visualization dashboards using frameworks such as React, Vue.js, or Angular',
 'Experienced in API development leveraging REST and GraphQL APIs for data services',
 'Strong grasp of microservices architecture and architectural design patterns',
 'Comprehensive understanding of CI/CD practices and DevOps responsibilities related to deploying and maintaining production software',
 'Knowledgeable in implementing and integrating artificial intelligence, machine learning, and related data solutions']

In [23]:
links = collection.query(query_texts=job['Data Engineer, ITC']['skills'], n_results=2).get('metadatas', [])
links

[[{'links': 'https://example.com/python-portfolio'},
  {'links': 'https://example.com/ml-python-portfolio'}],
 [{'links': 'https://example.com/flutter-portfolio'},
  {'links': 'https://example.com/kotlin-android-portfolio'}],
 [{'links': 'https://example.com/ml-python-portfolio'},
  {'links': 'https://example.com/ios-portfolio'}],
 [{'links': 'https://example.com/kotlin-backend-portfolio'},
  {'links': 'https://example.com/flutter-portfolio'}],
 [{'links': 'https://example.com/full-stack-js-portfolio'},
  {'links': 'https://example.com/react-portfolio'}],
 [{'links': 'https://example.com/vue-portfolio'},
  {'links': 'https://example.com/react-portfolio'}],
 [{'links': 'https://example.com/flutter-portfolio'},
  {'links': 'https://example.com/vue-portfolio'}],
 [{'links': 'https://example.com/android-portfolio'},
  {'links': 'https://example.com/xamarin-portfolio'}],
 [{'links': 'https://example.com/devops-portfolio'},
  {'links': 'https://example.com/magento-portfolio'}],
 [{'links': '

In [29]:
# prompt_email = PromptTemplate.from_template(
#         """
#         ### JOB DESCRIPTION:
#         {job_description}
        
#         ### INSTRUCTION:
#         You are Shashikala S, a 7th-semester B.E. student in Artificial Intelligence and Data Science at GSSSIETW.
#         You have strong skills in coding, emerging technologies, communication, team collaboration, leadership, and public speaking.

#         Write a cold email to a recruiter or hiring manager expressing interest in an internship or entry-level opportunity (without mentioning any specific job role).
#         The email should highlight your technical skills and tech stack relevant to the company or project (based on the links provided by the user).
#         Keep the tone professional, confident, and concise, showing enthusiasm for learning, innovation, and contributing to impactful projects
#         Do not provide a preamble.
#         ### EMAIL (NO PREAMBLE):
        
#         """
#         )

# chain_email = prompt_email | llm
# res = chain_email.invoke({"job_description": str(job), "link_list": links})
# print(res.content)

In [28]:
from langchain.prompts import PromptTemplate

prompt_email = PromptTemplate.from_template(
    """
    ### JOB DESCRIPTION:
    {job_description}
    
    ### LINKS / PORTFOLIO REFERENCES:
    {link_list}
    
    ### INSTRUCTION:
    You are Shashikala S, a 7th-semester B.E. student in Artificial Intelligence and Data Science at GSSSIETW.
    You have strong skills in coding, emerging technologies, communication, team collaboration, leadership, and public speaking.

    Write a cold email to a recruiter or hiring manager expressing interest in an internship or entry-level opportunity (without mentioning any specific job role).
    The email should highlight your technical skills and tech stack relevant to the company or project (based on the links provided by the user).
    Naturally incorporate the most relevant portfolio links ({link_list}) to demonstrate alignment with the company’s goals or technologies.
    Keep the tone professional, confident, and concise, showing enthusiasm for learning, innovation, and contributing to impactful projects.
    
    Do not provide a preamble.

    ### EMAIL (NO PREAMBLE):
    """
)

chain_email = prompt_email | llm

res = chain_email.invoke({
    "job_description": str(job),
        "link_list": links
})

print(res.content)


Subject: Expressed Interest in Internship or Entry-Level Opportunity

Dear Hiring Manager,

I am reaching out to express my interest in exploring internship or entry-level opportunities at your esteemed organization. As a 7th-semester student in Artificial Intelligence and Data Science at GSSSIETW, I am confident that my skills and passion for emerging technologies align with your company's vision.

Throughout my academic journey, I have developed a strong foundation in coding, particularly in languages such as Python, Java, and Kotlin. I have also gained practical experience in machine learning, data science, and full-stack development using frameworks like React, Vue.js, and Angular. Additionally, I have explored mobile app development using Flutter, Kotlin, and React Native.

I am impressed by your company's commitment to innovation and leveraging technologies like Databricks, AWS, and Snowflake. I am excited to learn more about your projects and how my skills can contribute to your